# Activation Functions and Loss Functions

A neural network layer computes $\mathbf{z} = \mathbf{W}\mathbf{a} + \mathbf{b}$, then applies an **activation function** to produce $\mathbf{a} = f(\mathbf{z})$. The activation introduces nonlinearity — without it, stacking layers would be pointless. The **loss function** measures how wrong the network's prediction is compared to the true target. Training minimizes this loss.

Choosing the right activation for each layer and the right loss for each task is not arbitrary — specific pairings are designed to work together mathematically and numerically. This notebook covers the major activation and loss functions in depth: their formulas, derivatives, behavior, and when to use each.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. Why Activation Functions Matter

Without activation functions, a chain of linear transformations is still linear. Two layers: $\mathbf{W}_2(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2 = \mathbf{W}'\mathbf{x} + \mathbf{b}'$ — equivalent to a single layer. No matter how many layers, the network can only learn linear mappings.

A nonlinear activation $f$ breaks this limitation:

$$\mathbf{a} = f(\mathbf{z}) = f(\mathbf{W}\mathbf{x} + \mathbf{b})$$

Each layer warps the input space differently. Composing these warps creates the ability to approximate complex, nonlinear functions.

Activation functions must be:
- **Nonlinear** — or the network collapses to linearity.
- **Differentiable** (almost everywhere) — so gradients can flow during backpropagation.
- **Computationally cheap** — called billions of times during training.
- **Well-behaved gradients** — neither vanishing to zero nor exploding to infinity in typical ranges.

---

## 2. Sigmoid

The **sigmoid** (logistic) function squashes any real number into the range $(0, 1)$:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Derivative:**

$$\sigma'(z) = \sigma(z)(1 - \sigma(z))$$

The derivative is convenient — it can be computed from the output itself without storing $z$.

**Properties:**
- Output interpretable as probability (for binary classification output layers).
- Smooth, differentiable everywhere.
- **Not zero-centered** — outputs are always positive, causing zigzag gradient updates.
- **Vanishing gradient** — for large $|z|$, $\sigma'(z) \approx 0$. In deep networks, gradients shrink as they propagate backward through sigmoid layers.

**Use today:** Output layer for binary classification. Rarely used in hidden layers (ReLU replaced it).

---

## 3. Hyperbolic Tangent (Tanh)

Tanh maps inputs to the range $(-1, 1)$:

$$\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$$

**Derivative:**

$$\tanh'(z) = 1 - \tanh^2(z)$$

**Relationship to sigmoid:** $\tanh(z) = 2\sigma(2z) - 1$. Tanh is a scaled and shifted sigmoid.

**Advantages over sigmoid:** Zero-centered outputs — gradients tend to move in consistent directions.

**Disadvantages:** Still suffers from vanishing gradients for large $|z|$. Largely replaced by ReLU in hidden layers, but still used in RNNs (LSTM gates) and some output layers.

In [ ]:
# Sigmoid and Tanh: functions and derivatives
z = np.linspace(-6, 6, 300)

sigmoid = 1 / (1 + np.exp(-z))
sigmoid_grad = sigmoid * (1 - sigmoid)
tanh = np.tanh(z)
tanh_grad = 1 - tanh**2

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(z, sigmoid, "b-", linewidth=2)
axes[0, 0].set_title("Sigmoid σ(z)")
axes[0, 0].axhline(0, color="gray", linewidth=0.5)
axes[0, 0].axvline(0, color="gray", linewidth=0.5)

axes[0, 1].plot(z, sigmoid_grad, "b-", linewidth=2)
axes[0, 1].set_title("Sigmoid derivative (max = 0.25)")
axes[0, 1].axhline(0, color="gray", linewidth=0.5)

axes[1, 0].plot(z, tanh, "g-", linewidth=2)
axes[1, 0].set_title("Tanh(z)")
axes[1, 0].axhline(0, color="gray", linewidth=0.5)
axes[1, 0].axvline(0, color="gray", linewidth=0.5)

axes[1, 1].plot(z, tanh_grad, "g-", linewidth=2)
axes[1, 1].set_title("Tanh derivative (max = 1.0)")
axes[1, 1].axhline(0, color="gray", linewidth=0.5)

for ax in axes.flat:
    ax.set_xlabel("z")

plt.tight_layout()
plt.show()

---

## 4. ReLU and Variants

### 4.1 ReLU (Rectified Linear Unit)

$$\text{ReLU}(z) = \max(0, z) = \begin{cases} z & z > 0 \\ 0 & z \leq 0 \end{cases}$$

**Derivative:**

$$\text{ReLU}'(z) = \begin{cases} 1 & z > 0 \\ 0 & z \leq 0 \end{cases}$$

**Why ReLU dominates:**
- Computationally trivial (`max(0, z)`).
- Does not saturate for $z > 0$ — gradient is 1, no vanishing gradient in the positive region.
- Induces sparsity — negative inputs produce zero activations.
- Empirically trains faster than sigmoid/tanh.

**Problem — dying ReLU:** If a neuron's weights push $z \leq 0$ for all training inputs, the neuron always outputs 0, gradient is always 0, and it never recovers. The neuron is "dead."

### 4.2 Leaky ReLU

$$\text{LeakyReLU}(z) = \begin{cases} z & z > 0 \\ \alpha z & z \leq 0 \end{cases}$$

Small slope $\alpha$ (typically 0.01) for negative inputs prevents neurons from dying completely.

### 4.3 Other Variants

- **PReLU** — $\alpha$ is learned per neuron during training.
- **ELU** — smooth curve for $z < 0$, approaches $-\alpha$ asymptotically. Mean activations closer to zero.
- **GELU** — used in Transformers (BERT, GPT): $\text{GELU}(z) = z \cdot \Phi(z)$ where $\Phi$ is the Gaussian CDF. Smooth, non-monotonic.
- **Swish** — $z \cdot \sigma(z)$. Self-gated, smooth.

In [ ]:
# ReLU family
z = np.linspace(-4, 4, 300)

relu = np.maximum(0, z)
leaky_relu = np.where(z > 0, z, 0.01 * z)
elu = np.where(z > 0, z, np.exp(z) - 1)
gelu = z * (1 / (1 + np.exp(-1.702 * z)))  # approximation

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

for ax, fn, name in zip(axes,
    [relu, leaky_relu, elu, gelu],
    ["ReLU", "Leaky ReLU (α=0.01)", "ELU", "GELU"]):
    ax.plot(z, fn, linewidth=2)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.axvline(0, color="gray", linewidth=0.5)
    ax.set_title(name)
    ax.set_xlabel("z")

plt.tight_layout()
plt.show()

In [ ]:
# ReLU derivative vs sigmoid derivative — why ReLU trains faster
z = np.linspace(-4, 4, 300)
relu_grad = (z > 0).astype(float)
sigmoid_grad = (1 / (1 + np.exp(-z))) * (1 - 1 / (1 + np.exp(-z)))

plt.figure(figsize=(8, 4))
plt.plot(z, relu_grad, "r-", linewidth=2, label="ReLU' (0 or 1)")
plt.plot(z, sigmoid_grad, "b-", linewidth=2, label="Sigmoid' (max 0.25)")
plt.xlabel("z")
plt.ylabel("Gradient")
plt.title("ReLU passes full gradient; sigmoid attenuates it")
plt.legend()
plt.show()

---

## 5. Softmax

Softmax converts a vector of raw scores (logits) into a probability distribution over $K$ classes:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

Properties:
- Each output is in $(0, 1)$.
- Outputs sum to 1.
- Amplifies differences — the largest logit gets disproportionately more probability.

**Numerical stability:** Compute $\text{softmax}(z_i - \max(z))$ to prevent overflow from large exponentials.

Softmax is used at the **output layer** for multi-class classification, almost always paired with **categorical cross-entropy loss**.

In [ ]:
def softmax(z):
    z_shifted = z - np.max(z, axis=0, keepdims=True)
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z, axis=0, keepdims=True)

# Effect of temperature on softmax
logits = np.array([2.0, 1.0, 0.5, 0.1])
temperatures = [0.5, 1.0, 2.0, 5.0]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
classes = ["Cat", "Dog", "Bird", "Fish"]

for ax, T in zip(axes, temperatures):
    probs = softmax(logits / T)
    ax.bar(classes, probs, color="steelblue", alpha=0.8)
    ax.set_ylim(0, 1)
    ax.set_title(f"Temperature T={T}")
    ax.tick_params(axis="x", rotation=30)

plt.suptitle("Higher temperature → softer (more uniform) distribution", y=1.02)
plt.tight_layout()
plt.show()

---

## 6. Linear (Identity) Activation

$$f(z) = z$$

No transformation. Used at the **output layer for regression** — the network predicts a continuous value directly. Also used internally in residual connections (skip connections add the input without activation).

Hidden layers with linear activation are pointless (composition of linear = linear), but output linear layers are essential for regression tasks.

---

## 7. Choosing Activation Functions

| Layer | Recommended | Avoid |
|-------|------------|-------|
| Hidden layers (default) | ReLU | Sigmoid, Tanh (vanishing gradients) |
| Hidden layers (Transformers) | GELU | — |
| Binary output | Sigmoid | ReLU (unbounded) |
| Multi-class output | Softmax | Sigmoid (doesn't sum to 1) |
| Regression output | Linear (identity) | Sigmoid (bounded to 0-1) |
| RNN gates (LSTM/GRU) | Sigmoid, Tanh | ReLU |

When in doubt: **ReLU for hidden layers, match output activation to your task.**

---

## 8. What Is a Loss Function?

The **loss function** (cost function) measures the discrepancy between the network's prediction $\hat{y}$ and the true target $y$. Training minimizes the average loss over the training set:

$$\mathcal{L} = \frac{1}{m} \sum_{i=1}^{m} L(\hat{y}_i, y_i)$$

The choice of loss depends on the task:

- **Regression** — how far is the prediction from the true value?
- **Classification** — how wrong is the predicted probability distribution?

The loss must be differentiable so gradients can flow back to update weights. The gradient of the loss with respect to the network's output is the starting point of backpropagation.

---

## 9. Regression Loss Functions

### 9.1 Mean Squared Error (MSE)

$$L_{\text{MSE}} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**Gradient:** $\frac{\partial L}{\partial \hat{y}_i} = -\frac{2}{n}(y_i - \hat{y}_i)$

Penalizes large errors heavily (squaring). Sensitive to outliers. Default for regression with linear output.

### 9.2 Mean Absolute Error (MAE)

$$L_{\text{MAE}} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

Robust to outliers. Not differentiable at zero, but subgradient works in practice.

### 9.3 Huber Loss

Combines MSE (for small errors) and MAE (for large errors). Less sensitive to outliers than MSE, smoother than MAE.

In [ ]:
# Regression loss comparison
y_true = 2.0
y_pred = np.linspace(-2, 6, 200)

mse = (y_true - y_pred)**2
mae = np.abs(y_true - y_pred)
huber_delta = 1.0
huber = np.where(np.abs(y_true - y_pred) <= huber_delta,
                 0.5 * (y_true - y_pred)**2,
                 huber_delta * (np.abs(y_true - y_pred) - 0.5 * huber_delta))

plt.figure(figsize=(8, 4))
plt.plot(y_pred, mse, label="MSE", linewidth=2)
plt.plot(y_pred, mae, label="MAE", linewidth=2)
plt.plot(y_pred, huber, label="Huber (δ=1)", linewidth=2)
plt.axvline(y_true, color="gray", linestyle="--", alpha=0.5, label="True value")
plt.xlabel("Predicted value ŷ")
plt.ylabel("Loss")
plt.title(f"Regression losses (true y = {y_true})")
plt.legend()
plt.show()

---

## 10. Binary Cross-Entropy Loss

For binary classification with sigmoid output $\hat{p} \in (0, 1)$ and true label $y \in \{0, 1\}$:

$$L = -\left[ y \log(\hat{p}) + (1 - y) \log(1 - \hat{p}) \right]$$

Intuition:
- If $y = 1$: loss = $-\log(\hat{p})$. Low loss when $\hat{p}$ is close to 1; loss → ∞ as $\hat{p}$ → 0.
- If $y = 0$: loss = $-\log(1 - \hat{p})$. Low loss when $\hat{p}$ is close to 0.

This loss heavily penalizes confident wrong predictions — exactly what we want for classification.

**Gradient (combined with sigmoid):** When sigmoid output and binary cross-entropy are used together, the gradient simplifies elegantly to:

$$\frac{\partial L}{\partial z} = \hat{p} - y$$

where $z$ is the pre-sigmoid logit. This clean gradient is why sigmoid + BCE is the standard binary classification pairing.

In [ ]:
def binary_cross_entropy(y, p_hat, eps=1e-15):
    p_hat = np.clip(p_hat, eps, 1 - eps)
    return -(y * np.log(p_hat) + (1 - y) * np.log(1 - p_hat))

p = np.linspace(0.001, 0.999, 200)

plt.figure(figsize=(8, 4))
plt.plot(p, binary_cross_entropy(1, p), "b-", linewidth=2, label="y = 1")
plt.plot(p, binary_cross_entropy(0, p), "r-", linewidth=2, label="y = 0")
plt.xlabel("Predicted probability ŷ")
plt.ylabel("Loss")
plt.title("Binary Cross-Entropy")
plt.legend()
plt.ylim(0, 5)
plt.show()

---

## 11. Categorical Cross-Entropy Loss

For multi-class classification with $K$ classes, true label as one-hot vector $\mathbf{y}$ and predicted probabilities $\hat{\mathbf{p}}$ from softmax:

$$L = -\sum_{k=1}^{K} y_k \log(\hat{p}_k)$$

Since only one $y_k = 1$ (the correct class), this simplifies to:

$$L = -\log(\hat{p}_{\text{correct class}})$$

The loss is low when the network assigns high probability to the correct class.

**Gradient (softmax + cross-entropy combined):**

$$\frac{\partial L}{\partial z_k} = \hat{p}_k - y_k$$

Again, a beautifully simple gradient — the difference between predicted and true probability. This is why softmax and categorical cross-entropy are always paired.

In [ ]:
def categorical_cross_entropy(y_true_idx, logits):
    """y_true_idx: integer class index. logits: (K,) array."""
    probs = softmax(logits.reshape(-1, 1)).flatten()
    return -np.log(probs[y_true_idx] + 1e-15)

# Loss as a function of the correct class probability
correct_probs = np.linspace(0.01, 0.99, 100)
losses = -np.log(correct_probs)

plt.figure(figsize=(8, 4))
plt.plot(correct_probs, losses, "b-", linewidth=2)
plt.xlabel("Predicted probability of correct class")
plt.ylabel("Cross-entropy loss")
plt.title("Categorical CE: loss → ∞ as confidence in correct class → 0")
plt.show()

# Example: 3-class problem
logits = np.array([2.0, 1.0, 0.1])
true_class = 0
print(f"Logits: {logits}")
print(f"Softmax: {np.round(softmax(logits.reshape(-1,1)).flatten(), 3)}")
print(f"Loss (true class={true_class}): {categorical_cross_entropy(true_class, logits):.4f}")

---

## 12. Pairing Activations with Loss Functions

The output activation and loss function must be chosen together:

| Task | Output Activation | Loss Function |
|------|------------------|---------------|
| Binary classification | Sigmoid | Binary cross-entropy |
| Multi-class classification | Softmax | Categorical cross-entropy |
| Regression | Linear (identity) | MSE or MAE |
| Multi-label classification | Sigmoid (per output) | Binary cross-entropy (per label) |

Using the wrong pairing causes problems:
- **Sigmoid + MSE** for classification — gradients are weak when predictions are confident and wrong.
- **Softmax + MSE** — does not penalize probability distributions correctly.
- **Linear + cross-entropy** — cross-entropy requires probabilities in $(0,1)$.

Frameworks often combine the final activation and loss into a single operation for numerical stability (e.g., `BCEWithLogitsLoss` in PyTorch applies sigmoid + BCE without computing intermediate probabilities).

---

## 13. Other Loss Functions

**Hinge Loss** — used with SVMs and some neural classifiers. $L = \max(0, 1 - y \cdot \hat{y})$ where $y \in \{-1, +1\}$. Penalizes predictions within the margin.

**Focal Loss** — modification of cross-entropy that down-weights easy examples and focuses on hard ones. Effective for imbalanced object detection.

**KL Divergence** — measures difference between two probability distributions. Used in variational autoencoders and knowledge distillation.

**Contrastive / Triplet Loss** — used in embedding learning. Pulls similar examples together, pushes dissimilar ones apart in embedding space.

**Perceptual Loss** — compares high-level feature representations rather than pixel values. Used in image generation and style transfer.

---

## 14. Loss Gradients — The Starting Point of Learning

Training updates weights to reduce loss. The chain rule starts with the gradient of loss with respect to the network output:

| Pairing | $\frac{\partial L}{\partial z}$ (gradient w.r.t. pre-activation) |
|---------|-------------------------------------------------------------|
| Sigmoid + BCE | $\hat{p} - y$ |
| Softmax + CE | $\hat{p}_k - y_k$ |
| Linear + MSE | $\frac{2}{n}(\hat{y} - y)$ |

These simple gradients are not coincidental — the activation and loss are mathematical duals designed so their derivatives cancel complexity. This is why pairings matter.

From this output gradient, backpropagation applies the chain rule layer by layer to compute gradients for every weight and bias in the network.

In [ ]:
# Verify sigmoid + BCE combined gradient: dL/dz = p_hat - y
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = 1.5
y = 1.0
p_hat = sigmoid(z)

# Numerical gradient
eps = 1e-7
loss_plus = binary_cross_entropy(y, sigmoid(z + eps))
loss_minus = binary_cross_entropy(y, sigmoid(z - eps))
numerical_grad = (loss_plus - loss_minus) / (2 * eps)

# Analytical: dL/dz = p_hat - y (combined sigmoid+BCE)
analytical_grad = p_hat - y

print(f"z = {z}, y = {y}, σ(z) = {p_hat:.4f}")
print(f"Numerical dL/dz:   {numerical_grad:.6f}")
print(f"Analytical p̂ - y:  {analytical_grad:.6f}")

---

## 15. Summary

**Activation functions** introduce nonlinearity into neural networks. ReLU is the default for hidden layers — fast, avoids vanishing gradients in the positive region. Sigmoid and tanh are largely relegated to specific architectures (gates, binary outputs). GELU powers Transformers. Softmax converts logits to class probabilities.

**Loss functions** measure prediction error. MSE for regression. Binary cross-entropy for binary classification. Categorical cross-entropy for multi-class. The output activation and loss must be paired correctly — their combined gradients simplify to elegant forms like $\hat{p} - y$ that make learning efficient.

These functions define what the network optimizes and how gradients flow. Backpropagation uses the loss gradient as its starting point and propagates it backward through activations and weights to update the entire network.